In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install jiwer
!pip install einops addict easydict

In [ ]:
import os
import json
import torch
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict
import pandas as pd
import pickle

# Metrics
from jiwer import cer, wer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
import nltk

# Download NLTK data for METEOR (silently)
import warnings
warnings.filterwarnings('ignore')

try:
    nltk.data.find('wordnet')
except LookupError:
    nltk.download('wordnet', quiet=True)
    nltk.download('punkt', quiet=True)
    nltk.download('omw-1.4', quiet=True)

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download("unsloth/DeepSeek-OCR-2", local_dir = "deepseek_ocr2")

In [ ]:
# ========================================
# STEP 1: LOAD MODEL FROM SCRATCH
# ========================================
print("="*80)
print(" LOADING MODEL FROM SCRATCH")
print("="*80)

from unsloth import FastVisionModel
import torch
from transformers import AutoModel
from datasets import load_dataset
import os
os.environ["UNSLOTH_WARN_UNINITIALIZED"] = '0'

# Load base model
print("\n Loading base DeepSeek-OCR model...")
model, tokenizer = FastVisionModel.from_pretrained(
    "deepseek_ocr2",
    load_in_4bit=False,
    auto_model=AutoModel,
    trust_remote_code=True,
    unsloth_force_compile=True,
    use_gradient_checkpointing="unsloth",
)

In [ ]:
# ==================== LOAD FROM HUGGING FACE ====================

from datasets import load_dataset
from huggingface_hub import login
import pandas as pd
from PIL import Image

#  Login to Hugging Face (if private dataset)
login()  # Enter your token

In [ ]:
# ========================================
# STEP 2: LOAD TEST DATASET
# ========================================
print("\n" + "="*80)
print(" LOADING TEST DATASET")
print("="*80)

# Load your dataset
print("\nLoading dataset from Hugging Face...")
dataset = load_dataset("avishadilhara/sinhala-ocr-lk-acts-1010")

print(f" Dataset loaded!")
print(f"   Test samples: {len(dataset['test'])}")

In [ ]:
# ========================================
# CONFIGURATION
# ========================================
CHECKPOINT_INTERVAL = 10  # Save checkpoint every N samples
CHECKPOINT_FILE = "./test_results/checkpoint.pkl"

In [ ]:
# ========================================
# STEP 3: DEFINE METRICS FUNCTIONS
# ========================================

def calculate_anls(ground_truth, prediction, threshold=0.5):
    """Calculate Average Normalized Levenshtein Similarity (ANLS)"""
    try:
        from Levenshtein import distance as levenshtein_distance

        if len(ground_truth) == 0 and len(prediction) == 0:
            return 1.0
        if len(ground_truth) == 0 or len(prediction) == 0:
            return 0.0

        edit_distance = levenshtein_distance(ground_truth, prediction)
        max_len = max(len(ground_truth), len(prediction))
        normalized_distance = edit_distance / max_len

        if normalized_distance < threshold:
            anls = 1 - normalized_distance
        else:
            anls = 0.0

        return anls
    except ImportError:
        from jiwer import cer
        return 1 - cer(ground_truth, prediction)


def calculate_all_metrics(ground_truth, prediction):
    """Calculate all metrics: CER, WER, BLEU, ANLS, METEOR"""
    metrics = {}

    # CER (Character Error Rate) - Lower is better
    # CLAMP to [0, 1] range since jiwer can return > 1.0
    # when prediction is much longer than reference
    try:
        raw_cer = cer(ground_truth, prediction)
        metrics['CER'] = min(raw_cer, 1.0)  # Cap at 1.0 (100% error)
    except:
        metrics['CER'] = 1.0

    # WER (Word Error Rate) - Lower is better
    # Same issue: WER can exceed 1.0
    try:
        raw_wer = wer(ground_truth, prediction)
        metrics['WER'] = min(raw_wer, 1.0)  # Cap at 1.0
    except:
        metrics['WER'] = 1.0

    # BLEU Score - Higher is better (0-1) — already bounded
    try:
        reference = [list(ground_truth)]
        hypothesis = list(prediction)
        smoothing = SmoothingFunction().method1
        raw_bleu = sentence_bleu(reference, hypothesis,
                                        smoothing_function=smoothing)
        metrics['BLEU'] = max(0.0, min(raw_bleu, 1.0))
    except:
        metrics['BLEU'] = 0.0

    # ANLS - Higher is better — already bounded
    try:
        raw_anls = calculate_anls(ground_truth, prediction)
        metrics['ANLS'] = max(0.0, min(raw_anls, 1.0))
    except:
        metrics['ANLS'] = 0.0

    # METEOR - Higher is better (0-1) — already bounded
    try:
        reference_tokens = list(ground_truth)
        hypothesis_tokens = list(prediction)
        raw_meteor = meteor_score([reference_tokens], hypothesis_tokens)
        metrics['METEOR'] = max(0.0, min(raw_meteor, 1.0))
    except:
        metrics['METEOR'] = 0.0

    # Character Accuracy — now guaranteed 0-100%
    metrics['Char_Accuracy'] = max((1 - metrics['CER']) * 100, 0.0)

    return metrics

In [ ]:
# ========================================
# CHECKPOINT MANAGEMENT
# ========================================

def save_checkpoint(all_results, year_results, last_processed_idx, output_dir):
    """Save checkpoint to resume later"""
    checkpoint_data = {
        'all_results': all_results,
        'year_results': dict(year_results),
        'last_processed_idx': last_processed_idx
    }

    checkpoint_path = os.path.join(output_dir, "checkpoint.pkl")
    with open(checkpoint_path, 'wb') as f:
        pickle.dump(checkpoint_data, f)


def load_checkpoint(output_dir):
    """Load checkpoint if exists"""
    checkpoint_path = os.path.join(output_dir, "checkpoint.pkl")

    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, 'rb') as f:
            checkpoint_data = pickle.load(f)
        return checkpoint_data
    return None

In [ ]:
# ===============================================================================
# STEP 4: RUN TESTING ON ENTIRE TEST SET WITH CHECKPOINTING
# ===============================================================================

def test_entire_dataset(resume=True):
    """
    Test model on entire test dataset with checkpoint support.
    """

    # Create output directory
    output_dir = "./test_results"
    inference_dir = os.path.join(output_dir, "inference_outputs")
    os.makedirs(inference_dir, exist_ok=True)
    os.makedirs(os.path.join(inference_dir, "images"), exist_ok=True)

    # Results storage
    all_results = []
    year_results = defaultdict(list)
    start_idx = 0

    # CHECK FOR EXISTING CHECKPOINT
    if resume:
        checkpoint = load_checkpoint(output_dir)
        if checkpoint:
            all_results = checkpoint['all_results']
            year_results = defaultdict(list, checkpoint['year_results'])
            start_idx = checkpoint['last_processed_idx'] + 1

            print("="*80)
            print("CHECKPOINT FOUND - RESUMING FROM PREVIOUS RUN")
            print("="*80)
            print(f"Already processed: {len(all_results)} samples")
            print(f"Resuming from sample: {start_idx}")
            print("="*80)

    # Get test dataset
    test_dataset = dataset['test']
    total_samples = len(test_dataset)

    if start_idx == 0:
        print("="*80)
        print("TESTING MODEL ON ENTIRE TEST DATASET")
        print("="*80)
        print(f"Total samples to test: {total_samples}")
        print(f"Output directory: {inference_dir}")
        print(f"Checkpoint interval: Every {CHECKPOINT_INTERVAL} samples")
        print("="*80)

    # Test each sample
    samples_to_process = range(start_idx, total_samples)

    with tqdm(total=total_samples, initial=start_idx, desc="Testing", unit="sample", ncols=100) as pbar:
        for idx in samples_to_process:
            try:
                # Get sample
                sample = test_dataset[idx]

                # Get metadata
                ground_truth = sample['text']
                year = sample.get('year', 'Unknown')
                image = sample['image']

                # Convert RGBA to RGB before saving
                if image.mode == 'RGBA':
                    # Create white background
                    rgb_image = Image.new('RGB', image.size, (255, 255, 255))
                    rgb_image.paste(image, mask=image.split()[3])  # Use alpha channel as mask
                    image = rgb_image
                elif image.mode != 'RGB':
                    # Convert any other mode to RGB
                    image = image.convert('RGB')

                # Save image with unique name
                img_filename = f"{idx:04d}.jpg"
                temp_img_path = os.path.join(inference_dir, "images", img_filename)
                image.save(temp_img_path)

                # Run inference
                model.infer(
                    tokenizer,
                    prompt="<image>\nFree OCR.",
                    image_file=temp_img_path,
                    output_path=inference_dir,
                    base_size=1024,
                    image_size=768,
                    crop_mode=True,
                    save_results=True
                )

                # Read prediction from .mmd file
                prediction = ""
                mmd_file = os.path.join(inference_dir, "result.mmd")

                if os.path.exists(mmd_file):
                    with open(mmd_file, 'r', encoding='utf-8') as f:
                        prediction = f.read().strip()

                    # Rename to keep it
                    saved_mmd = os.path.join(inference_dir, f"result_{idx:04d}.mmd")
                    os.rename(mmd_file, saved_mmd)
                else:
                    # Alternative: check for other .mmd files
                    for file in os.listdir(inference_dir):
                        if file.endswith(".mmd"):
                            mmd_path = os.path.join(inference_dir, file)
                            with open(mmd_path, 'r', encoding='utf-8') as f:
                                prediction = f.read().strip()

                            saved_mmd = os.path.join(inference_dir, f"result_{idx:04d}.mmd")
                            os.rename(mmd_path, saved_mmd)
                            break

                # Calculate metrics
                metrics = calculate_all_metrics(ground_truth, prediction)

                # Store results
                result = {
                    'sample_idx': idx,
                    'year': year,
                    'ground_truth_length': len(ground_truth),
                    'prediction_length': len(prediction),
                    **metrics
                }

                all_results.append(result)
                year_results[year].append(result)

                # Update progress bar with current metrics
                pbar.set_postfix({
                    'CER': f"{metrics['CER']:.3f}",
                    'Acc': f"{metrics['Char_Accuracy']:.1f}"
                })
                pbar.update(1)

                # Save checkpoint periodically
                if (idx + 1) % CHECKPOINT_INTERVAL == 0:
                    save_checkpoint(all_results, year_results, idx, output_dir)
                    pbar.write(f"Checkpoint saved at sample {idx + 1}")

            except KeyboardInterrupt:
                print("\nTesting interrupted by user!")
                print("Saving checkpoint...")
                save_checkpoint(all_results, year_results, idx - 1, output_dir)
                print(f"Progress saved. Resume by running the code again.")
                print(f"Processed {len(all_results)}/{total_samples} samples")
                raise

            except Exception as e:
                # Log error quietly, don't print full traceback
                pbar.write(f"Error at sample {idx}: {str(e)[:50]}...")

                # Add failed result
                all_results.append({
                    'sample_idx': idx,
                    'year': year if 'year' in locals() else 'Unknown',
                    'CER': 1.0,
                    'WER': 1.0,
                    'BLEU': 0.0,
                    'ANLS': 0.0,
                    'METEOR': 0.0,
                    'Char_Accuracy': 0.0,
                    'ground_truth_length': 0,
                    'prediction_length': 0
                })
                pbar.update(1)
                continue

    # Save final checkpoint
    save_checkpoint(all_results, year_results, total_samples - 1, output_dir)
    print("\nAll samples processed!")

    return all_results, year_results


In [ ]:
# ========================================
# STEP 5: DISPLAY RESULTS
# ========================================

def display_results(all_results, year_results):
    """Display comprehensive results"""
    print("\n" + "="*80)
    print(" TEST RESULTS")
    print("="*80)

    # Convert to DataFrame
    df = pd.DataFrame(all_results)

    # ========================================
    # 1. OVERALL AVERAGE METRICS
    # ========================================
    print("\n" + "="*80)
    print("OVERALL AVERAGE METRICS")
    print("="*80)

    metrics_to_show = ['CER', 'WER', 'BLEU', 'ANLS', 'METEOR', 'Char_Accuracy']

    for metric in metrics_to_show:
        if metric in df.columns:
            avg_value = df[metric].mean()

            if metric in ['CER', 'WER']:
                direction = "Lower is better"
            else:
                direction = "Higher is better"

            print(f"{metric:15}: {avg_value:7.4f}  ({direction})")

    print(f"\n{'Total Samples':15}: {len(df)}")

    # ========================================
    # 2. YEAR-WISE AVERAGE METRICS
    # ========================================
    print("\n" + "="*80)
    print(" YEAR-WISE AVERAGE METRICS")
    print("="*80)

    # Group by year
    year_groups = df.groupby('year')

    # Create year summary table
    year_summary = []

    for year, group in sorted(year_groups):
        year_data = {
            'Year': year,
            'Samples': len(group),
            'CER': f"{group['CER'].mean():.4f}",
            'WER': f"{group['WER'].mean():.4f}",
            'BLEU': f"{group['BLEU'].mean():.4f}",
            'ANLS': f"{group['ANLS'].mean():.4f}",
            'METEOR': f"{group['METEOR'].mean():.4f}",
            'Accuracy': f"{group['Char_Accuracy'].mean():.2f}%"
        }
        year_summary.append(year_data)

    year_df = pd.DataFrame(year_summary)

    # Display year summary table
    print("\n" + year_df.to_string(index=False))

    # ========================================
    # 3. INDIVIDUAL SAMPLE RESULTS (First 10)
    # ========================================
    print("\n" + "="*80)
    print("INDIVIDUAL SAMPLE RESULTS (First 10 samples)")
    print("="*80)

    display_cols = ['sample_idx', 'year', 'CER', 'WER', 'BLEU', 'ANLS', 'METEOR', 'Char_Accuracy']
    print("\n" + df[display_cols].head(10).to_string(index=False))

    # ========================================
    # 4. BEST AND WORST PERFORMING SAMPLES
    # ========================================
    print("\n" + "="*80)
    print(" BEST PERFORMING SAMPLES (Top 5 by Character Accuracy)")
    print("="*80)

    best_samples = df.nlargest(5, 'Char_Accuracy')[display_cols]
    print("\n" + best_samples.to_string(index=False))

    print("\n" + "="*80)
    print(" WORST PERFORMING SAMPLES (Bottom 5 by Character Accuracy)")
    print("="*80)

    worst_samples = df.nsmallest(5, 'Char_Accuracy')[display_cols]
    print("\n" + worst_samples.to_string(index=False))

    # ========================================
    # 5. SAVE RESULTS TO CSV
    # ========================================
    output_csv = "./test_results/test_metrics.csv"
    df.to_csv(output_csv, index=False, encoding='utf-8')
    print(f"\nFull results saved to: {output_csv}")

    year_csv = "./test_results/year_wise_metrics.csv"
    year_df.to_csv(year_csv, index=False)
    print(f"Year-wise results saved to: {year_csv}")

    # ========================================
    # 6. SUMMARY STATISTICS
    # ========================================
    print("\n" + "="*80)
    print(" SUMMARY STATISTICS")
    print("="*80)

    print(f"\nSamples with >90% accuracy: {len(df[df['Char_Accuracy'] > 90])}")
    print(f"Samples with >80% accuracy: {len(df[df['Char_Accuracy'] > 80])}")
    print(f"Samples with <50% accuracy: {len(df[df['Char_Accuracy'] < 50])}")

    print(f"\nMedian Character Accuracy: {df['Char_Accuracy'].median():.2f}%")
    print(f"Std Dev Character Accuracy: {df['Char_Accuracy'].std():.2f}%")

    return df, year_df


In [ ]:
# ========================================
# STEP 6: RUN TESTING
# ========================================

if __name__ == "__main__":
    print("\n" + "="*80)
    print(" STARTING COMPREHENSIVE MODEL TESTING")
    print("="*80)

    try:
        # Run testing with resume support
        all_results, year_results = test_entire_dataset(resume=True)

        # Display results
        results_df, year_df = display_results(all_results, year_results)

        print("\n" + "="*80)
        print(" TESTING COMPLETED!")
        print("="*80)
        print(f"\nCheck ./test_results/ directory for:")
        print(f"   - test_metrics.csv (all samples)")
        print(f"   - year_wise_metrics.csv (year summary)")
        print(f"   - inference_outputs/ (model predictions as .mmd files)")
        print(f"   - checkpoint.pkl (resume checkpoint)")

    except KeyboardInterrupt:
        print("\n\n  Testing paused. Run again to resume from checkpoint.")


In [ ]:
import shutil
import os
from IPython.display import FileLink

# Create zip file of test_results folder
output_zip = 'test_results.zip'

if os.path.exists('./test_results'):
    # Remove existing zip if it exists
    if os.path.exists(output_zip):
        os.remove(output_zip)

    # Create zip archive
    shutil.make_archive('test_results', 'zip', './test_results')

    print(f"Created {output_zip}")
    print(f"Size: {os.path.getsize(output_zip) / (1024*1024):.2f} MB")

    # Provide download link
    display(FileLink(output_zip))
else:
    print("test_results folder not found")


In [ ]:
import pandas as pd

# ============================================================================
# LOAD EXISTING RESULTS AND FIX ALL METRICS TO 0-100% RANGE
# ============================================================================

print("="*80)
print("LOADING EXISTING TEST RESULTS")
print("="*80)

# Load your existing results
df = pd.read_csv('./test_results/test_metrics.csv')

print(f"Loaded {len(df)} samples")
print(f"\nBEFORE FIXING:")
print(f"ANLS Range: {df['ANLS'].min():.4f} to {df['ANLS'].max():.4f}")
print(f"BLEU Range: {df['BLEU'].min():.4f} to {df['BLEU'].max():.4f}")
print(f"METEOR Range: {df['METEOR'].min():.4f} to {df['METEOR'].max():.4f}")

# ============================================================================
# APPLY FIXES
# ============================================================================

print("\n" + "="*80)
print("APPLYING FIXES TO ALL METRICS")
print("="*80)

# Fix ANLS: Clamp negative values to 0, upper bound to 1
df['ANLS'] = df['ANLS'].clip(lower=0.0, upper=1.0)

# Fix BLEU: Ensure 0-1 range
df['BLEU'] = df['BLEU'].clip(lower=0.0, upper=1.0)

# Fix METEOR: Ensure 0-1 range
df['METEOR'] = df['METEOR'].clip(lower=0.0, upper=1.0)

# Fix CER and WER: Ensure 0-1 range
df['CER'] = df['CER'].clip(lower=0.0, upper=1.0)
df['WER'] = df['WER'].clip(lower=0.0, upper=1.0)

# Fix Character Accuracy: Ensure 0-100%
df['Char_Accuracy'] = df['Char_Accuracy'].clip(lower=0.0, upper=100.0)

print("\nAll metrics fixed!")

# ============================================================================
# SAVE CORRECTED RESULTS
# ============================================================================

df.to_csv('./test_results/test_metrics_corrected.csv', index=False)
print("\nSaved corrected results to: ./test_results/test_metrics_corrected.csv")

# ============================================================================
# DISPLAY FINAL RESULTS
# ============================================================================

print("\n" + "="*80)
print(" CORRECTED TEST RESULTS - ALL METRICS IN 0-100% RANGE")
print("="*80)

# 1. OVERALL AVERAGE METRICS
print("\n" + "="*80)
print("OVERALL AVERAGE METRICS (CORRECTED)")
print("="*80)

metrics_to_show = ['CER', 'WER', 'BLEU', 'ANLS', 'METEOR', 'Char_Accuracy']

for metric in metrics_to_show:
    if metric in df.columns:
        avg_value = df[metric].mean()

        if metric in ['CER', 'WER']:
            direction = "Lower is better"
        else:
            direction = "Higher is better"

        print(f"{metric:15}: {avg_value:7.4f} ({direction})")

print(f"\nTotal Samples   : {len(df)}")

# 2. YEAR-WISE AVERAGE METRICS
print("\n" + "="*80)
print("YEAR-WISE AVERAGE METRICS (CORRECTED)")
print("="*80)

year_groups = df.groupby('year')
year_summary = []

for year, group in sorted(year_groups):
    year_data = {
        'Year': year,
        'Samples': len(group),
        'CER': f"{group['CER'].mean():.4f}",
        'WER': f"{group['WER'].mean():.4f}",
        'BLEU': f"{group['BLEU'].mean():.4f}",
        'ANLS': f"{group['ANLS'].mean():.4f}",
        'METEOR': f"{group['METEOR'].mean():.4f}",
        'Accuracy': f"{group['Char_Accuracy'].mean():.2f}%"
    }
    year_summary.append(year_data)

year_df = pd.DataFrame(year_summary)
print("\n" + year_df.to_string(index=False))

# Save year-wise corrected results
year_df.to_csv('./test_results/year_wise_metrics_corrected.csv', index=False)
print("\nSaved year-wise corrected results to: ./test_results/year_wise_metrics_corrected.csv")

# 3. INDIVIDUAL SAMPLE RESULTS (First 10)
print("\n" + "="*80)
print("INDIVIDUAL SAMPLE RESULTS (First 10 samples) - CORRECTED")
print("="*80)

display_cols = ['sample_idx', 'year', 'CER', 'WER', 'BLEU', 'ANLS', 'METEOR', 'Char_Accuracy']
print("\n" + df[display_cols].head(10).to_string(index=False))

# 4. BEST AND WORST PERFORMING SAMPLES
print("\n" + "="*80)
print(" BEST PERFORMING SAMPLES (Top 5 by Character Accuracy)")
print("="*80)

best_samples = df.nlargest(5, 'Char_Accuracy')[display_cols]
print("\n" + best_samples.to_string(index=False))

print("\n" + "="*80)
print("  WORST PERFORMING SAMPLES (Bottom 5 by Character Accuracy)")
print("="*80)

worst_samples = df.nsmallest(5, 'Char_Accuracy')[display_cols]
print("\n" + worst_samples.to_string(index=False))

# 5. VALIDATION CHECK
print("\n" + "="*80)
print("VALIDATION: ALL METRICS IN VALID RANGE")
print("="*80)

validation_results = []
for metric in ['CER', 'WER', 'BLEU', 'ANLS', 'METEOR']:
    min_val = df[metric].min()
    max_val = df[metric].max()
    in_range = (min_val >= 0.0 and max_val <= 1.0)
    status = "PASS" if in_range else "FAIL"
    validation_results.append({
        'Metric': metric,
        'Min': f"{min_val:.4f}",
        'Max': f"{max_val:.4f}",
        'Expected Range': '0.0 - 1.0',
        'Status': status
    })

min_acc = df['Char_Accuracy'].min()
max_acc = df['Char_Accuracy'].max()
in_range_acc = (min_acc >= 0.0 and max_acc <= 100.0)
status_acc = "PASS" if in_range_acc else "FAIL"
validation_results.append({
    'Metric': 'Char_Accuracy',
    'Min': f"{min_acc:.2f}%",
    'Max': f"{max_acc:.2f}%",
    'Expected Range': '0% - 100%',
    'Status': status_acc
})

validation_df = pd.DataFrame(validation_results)
print("\n" + validation_df.to_string(index=False))

# 6. SUMMARY STATISTICS
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)

print(f"\nSamples with >90% accuracy: {len(df[df['Char_Accuracy'] > 90])}")
print(f"Samples with >80% accuracy: {len(df[df['Char_Accuracy'] > 80])}")
print(f"Samples with <50% accuracy: {len(df[df['Char_Accuracy'] < 50])}")

print(f"\nMedian Character Accuracy: {df['Char_Accuracy'].median():.2f}%")
print(f"Std Dev Character Accuracy: {df['Char_Accuracy'].std():.2f}%")

print("\n" + "="*80)
print("ALL METRICS NOW IN VALID 0-100% RANGE!")
print("="*80)
print("\nFiles saved:")
print("  - ./test_results/test_metrics_corrected.csv")
print("  - ./test_results/year_wise_metrics_corrected.csv")
print("="*80)
